# MicroFinML - Exploratory Data Analysis (EDA)
## Data-Driven Intelligence for Sustainable Economics: ML for Micro-Financial Growth

**Objective:** Explore and understand the Loan Default dataset to uncover patterns, distributions, and relationships that will guide our ML model development.

**Dataset:** Loan Default Dataset (255,347 records × 18 features)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import sys
import os

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

# Add project root to path
sys.path.insert(0, os.path.abspath('..'))

print('Libraries loaded successfully!')

## 1. Load and Inspect Data

In [ ]:
df = pd.read_csv('../data/raw/Loan Default.csv')
print(f'Dataset Shape: {df.shape}')
print(f'Total Records: {df.shape[0]:,}')
print(f'Total Features: {df.shape[1]}')
print(f'\nFirst 5 rows:')
df.head()

In [ ]:
print('=== Dataset Info ===')
df.info()
print(f'\nMemory Usage: {df.memory_usage(deep=True).sum() / 1e6:.2f} MB')

In [ ]:
print('=== Column Data Types ===')
print(f'\nNumeric columns ({df.select_dtypes(include=[np.number]).shape[1]}):')
print(list(df.select_dtypes(include=[np.number]).columns))
print(f'\nCategorical columns ({df.select_dtypes(include=["object"]).shape[1]}):')
print(list(df.select_dtypes(include=['object']).columns))

## 2. Missing Values & Data Quality

In [ ]:
missing = df.isnull().sum()
print('=== Missing Values ===')
print(f'Total missing values: {missing.sum()}')
print(f'\nMissing per column:')
print(missing[missing > 0] if missing.sum() > 0 else 'No missing values found! Dataset is clean.')

print(f'\n=== Duplicate Rows ===')
print(f'Duplicate rows: {df.duplicated().sum()}')

## 3. Statistical Summary

In [ ]:
print('=== Numeric Features - Statistical Summary ===')
df.describe().round(2)

In [ ]:
print('=== Categorical Features - Value Counts ===')
cat_cols = ['Education', 'EmploymentType', 'MaritalStatus', 'HasMortgage', 'HasDependents', 'LoanPurpose', 'HasCoSigner']
for col in cat_cols:
    print(f'\n--- {col} ---')
    print(df[col].value_counts())

## 4. Target Variable Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Count plot
target_counts = df['Default'].value_counts()
colors = ['#2196F3', '#FF5722']
axes[0].bar(['Repay (0)', 'Default (1)'], target_counts.values, color=colors, edgecolor='black')
axes[0].set_title('Loan Default Distribution', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Count')
for i, v in enumerate(target_counts.values):
    axes[0].text(i, v + 1000, f'{v:,}', ha='center', fontweight='bold', fontsize=12)

# Pie chart
axes[1].pie(target_counts.values, labels=['Repay (0)', 'Default (1)'],
            autopct='%1.1f%%', colors=colors, startangle=90,
            textprops={'fontsize': 13, 'fontweight': 'bold'},
            explode=(0, 0.05))
axes[1].set_title('Default Rate', fontsize=14, fontweight='bold')

plt.suptitle('Target Variable: Loan Default', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../results/figures/eda_plots/target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nDefault Rate: {df["Default"].mean():.2%}')
print(f'Class Imbalance Ratio: {target_counts[0]/target_counts[1]:.1f}:1')

## 5. Numeric Feature Distributions

In [ ]:
numeric_cols = ['Age', 'Income', 'LoanAmount', 'CreditScore', 'MonthsEmployed',
                'NumCreditLines', 'InterestRate', 'LoanTerm', 'DTIRatio']

fig, axes = plt.subplots(3, 3, figsize=(18, 14))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    ax = axes[i]
    # Plot distribution for default=0 and default=1
    df[df['Default'] == 0][col].hist(ax=ax, bins=40, alpha=0.6, label='Repay', color='#2196F3', density=True)
    df[df['Default'] == 1][col].hist(ax=ax, bins=40, alpha=0.6, label='Default', color='#FF5722', density=True)
    ax.set_title(f'{col}', fontsize=13, fontweight='bold')
    ax.legend(fontsize=9)
    ax.set_ylabel('Density')

plt.suptitle('Numeric Feature Distributions by Default Status', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/figures/eda_plots/numeric_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Box Plots - Outlier Detection

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(18, 14))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    ax = axes[i]
    sns.boxplot(data=df, x='Default', y=col, ax=ax, palette=['#2196F3', '#FF5722'])
    ax.set_title(f'{col} by Default Status', fontsize=13, fontweight='bold')
    ax.set_xticklabels(['Repay', 'Default'])

plt.suptitle('Box Plots: Numeric Features by Default Status', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/figures/eda_plots/boxplots.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Correlation Analysis

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))

corr_matrix = df[numeric_cols + ['Default']].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.3f', cmap='RdBu_r',
            center=0, square=True, linewidths=1, ax=ax,
            vmin=-1, vmax=1)

ax.set_title('Correlation Heatmap', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/figures/eda_plots/correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nCorrelation with Default (Target):')
print(corr_matrix['Default'].sort_values(ascending=False).drop('Default').to_string())

## 8. Categorical Features vs Default Rate

In [ ]:
cat_cols = ['Education', 'EmploymentType', 'MaritalStatus', 'HasMortgage',
            'HasDependents', 'LoanPurpose', 'HasCoSigner']

fig, axes = plt.subplots(3, 3, figsize=(20, 16))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    ax = axes[i]
    default_rate = df.groupby(col)['Default'].mean().sort_values(ascending=False)
    bars = ax.bar(range(len(default_rate)), default_rate.values, color='#FF5722', edgecolor='black', alpha=0.8)
    ax.set_xticks(range(len(default_rate)))
    ax.set_xticklabels(default_rate.index, rotation=45, ha='right', fontsize=10)
    ax.set_title(f'Default Rate by {col}', fontsize=13, fontweight='bold')
    ax.set_ylabel('Default Rate')
    ax.axhline(y=df['Default'].mean(), color='blue', linestyle='--', alpha=0.7, label=f'Overall: {df["Default"].mean():.2%}')
    ax.legend(fontsize=9)
    for bar, val in zip(bars, default_rate.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
                f'{val:.1%}', ha='center', fontsize=9, fontweight='bold')

# Hide unused subplots
for j in range(len(cat_cols), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Default Rate by Categorical Features', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/figures/eda_plots/categorical_default_rates.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Categorical Feature Distributions

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(20, 16))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    ax = axes[i]
    ct = pd.crosstab(df[col], df['Default'], normalize='index')
    ct.plot(kind='bar', stacked=True, ax=ax, color=['#2196F3', '#FF5722'], edgecolor='black')
    ax.set_title(f'{col}', fontsize=13, fontweight='bold')
    ax.set_ylabel('Proportion')
    ax.legend(['Repay', 'Default'], fontsize=9)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')

for j in range(len(cat_cols), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Categorical Feature Distributions (Stacked by Default)', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/figures/eda_plots/categorical_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Pairwise Relationships

In [ ]:
# Key feature relationships
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Income vs LoanAmount
sample = df.sample(5000, random_state=42)
axes[0, 0].scatter(sample[sample['Default']==0]['Income'], sample[sample['Default']==0]['LoanAmount'],
                    alpha=0.3, s=10, c='#2196F3', label='Repay')
axes[0, 0].scatter(sample[sample['Default']==1]['Income'], sample[sample['Default']==1]['LoanAmount'],
                    alpha=0.3, s=10, c='#FF5722', label='Default')
axes[0, 0].set_xlabel('Income')
axes[0, 0].set_ylabel('Loan Amount')
axes[0, 0].set_title('Income vs Loan Amount', fontsize=13, fontweight='bold')
axes[0, 0].legend()

# CreditScore vs InterestRate
axes[0, 1].scatter(sample[sample['Default']==0]['CreditScore'], sample[sample['Default']==0]['InterestRate'],
                    alpha=0.3, s=10, c='#2196F3', label='Repay')
axes[0, 1].scatter(sample[sample['Default']==1]['CreditScore'], sample[sample['Default']==1]['InterestRate'],
                    alpha=0.3, s=10, c='#FF5722', label='Default')
axes[0, 1].set_xlabel('Credit Score')
axes[0, 1].set_ylabel('Interest Rate')
axes[0, 1].set_title('Credit Score vs Interest Rate', fontsize=13, fontweight='bold')
axes[0, 1].legend()

# DTIRatio vs Default
sns.violinplot(data=df, x='Default', y='DTIRatio', ax=axes[1, 0], palette=['#2196F3', '#FF5722'])
axes[1, 0].set_xticklabels(['Repay', 'Default'])
axes[1, 0].set_title('DTI Ratio by Default Status', fontsize=13, fontweight='bold')

# CreditScore vs Default
sns.violinplot(data=df, x='Default', y='CreditScore', ax=axes[1, 1], palette=['#2196F3', '#FF5722'])
axes[1, 1].set_xticklabels(['Repay', 'Default'])
axes[1, 1].set_title('Credit Score by Default Status', fontsize=13, fontweight='bold')

plt.suptitle('Key Feature Relationships', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/figures/eda_plots/pairwise_relationships.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. Key Insights Summary

### Dataset Quality
- **No missing values** - dataset is clean and ready for modeling
- **255,347 records** with 17 predictive features + 1 target

### Class Imbalance
- **88.4% Repay vs 11.6% Default** - significant imbalance (~7.6:1 ratio)
- Will need SMOTE or class weights to handle this

### Feature Observations
- Mix of **9 numeric** and **7 categorical** features provides rich information
- Credit Score, Income, DTI Ratio, and Interest Rate are likely strong predictors
- Loan Purpose and Employment Type show variation in default rates

### Next Steps
1. Feature engineering (Income-to-Loan ratios, credit score groups, etc.)
2. Encode categorical variables (One-Hot Encoding)
3. Scale numeric features (StandardScaler)
4. Apply SMOTE for class imbalance
5. Train ML models (XGBoost, Random Forest, Logistic Regression)